# Lección 15.1 — Del notebook a producción: MLOps en una página

> **Módulo 15 · ML en producción y responsabilidad** · Nivel 5 — Héroe
> ⏱️ Tiempo estimado: 80–100 min (lección con ejercicios intercalados ≈ 55 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 1.5 (ciclo de vida de un proyecto), Lección 2.6 (fuga de datos), Lección 7.5 (pipelines)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar qué es **MLOps** y por qué el modelo es solo una pequeña parte de un sistema de ML en producción.
- Empaquetar un modelo como **pipeline**, guardarlo, cargarlo y servir predicciones con **validación de entradas**.
- Reconocer el **desajuste entre entrenamiento y producción** (*training-serving skew*) y otros errores típicos del despliegue.
- Escribir **pruebas de comportamiento** para un modelo y elegir entre predicción por lotes y en línea.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`).

In [ ]:
import json, time, tempfile, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

CARPETA = Path(tempfile.mkdtemp())     # carpeta temporal para "guardar en producción"
print("✅ Todo listo")

---
## 1. 📖 La intuición: de la cocina de casa a la cadena de restaurantes 🍝

Preparaste una salsa increíble en tu cocina. Ahora una cadena de 200 restaurantes quiere servirla **todos los días**. De pronto aparecen preguntas que en tu casa nunca te hiciste:

- ¿La receta está escrita con **exactamente** los mismos ingredientes y cantidades en todos los locales?
- ¿Qué pasa si un proveedor manda tomates **distintos**?
- ¿Cómo sabes si en el local 147 la salsa **salió mal** hoy?
- Si cambias la receta, ¿puedes **volver** a la anterior si a los clientes no les gusta?

Un modelo en un notebook es la salsa de tu casa. **MLOps** (*Machine Learning Operations*) es todo lo necesario para servirla a escala, de forma **confiable, repetible y vigilada**.

### ✋ Ejercicio 1 — Calentamiento
Entrenaste en tu notebook un modelo que predice qué clientes cancelarán su contrato, con AUC = 0.85. Lo conectan al sistema de la empresa. Escribe **tres** cosas que podrían salir mal en el primer mes, aunque el modelo sea "bueno".

✍️ *Mi lista:*

<details><summary><b>👀 Ver respuesta</b></summary>

Algunas posibilidades (hay muchas):
1. El sistema le manda los datos en **otro formato** o en otras unidades (pesos vs. miles de pesos, fechas en otro orden).
2. Llega un valor que nunca vio (un plan nuevo, un campo vacío) y el modelo **falla** o predice cualquier cosa.
3. Los clientes **cambian** (una promoción de la competencia) y el modelo deja de acertar sin que nadie se entere.
4. Nadie sabe **qué versión** del modelo está funcionando ni con qué datos se entrenó.
5. Es demasiado **lento** para responder mientras el cliente espera en la app.

Esta lección ataca 1, 2, 4 y 5. La 3 es el tema de las Lecciones 15.2 y 15.3.
</details>

---
## 2. 📖 El concepto: el modelo es la parte pequeña

Un famoso artículo de Google (*Hidden Technical Debt in Machine Learning Systems*, 2015) mostró que en un sistema real **el código del modelo es una cajita diminuta** rodeada de todo esto:

```
┌──────────────────────────────────────────────────────────────────────┐
│ recolección de datos · validación de datos · extracción de features  │
│ ┌──────────────┐                                                     │
│ │ CÓDIGO DEL   │  configuración · gestión de recursos · servidores   │
│ │   MODELO     │  versionado · registro de modelos · pruebas         │
│ └──────────────┘  monitoreo · alertas · reentrenamiento · rollback   │
└──────────────────────────────────────────────────────────────────────┘
```

### 📖 El ciclo de vida en producción

```
 datos ─► entrenar ─► validar ─► registrar ─► desplegar ─► servir predicciones
   ▲                   (¿supera     (versión,      (lotes o       │
   │                    al actual?)  métricas)      en línea)      ▼
   └──────────── reentrenar ◄────── alertas ◄────── MONITOREAR ◄──┘
```

**MLOps** = las prácticas y herramientas para que este ciclo sea **automático, reproducible y vigilado**. Es la versión ML de DevOps, con un problema extra: aunque **nadie toque el código**, el modelo puede empeorar porque **el mundo cambia**.

### ✋ Ejercicio 2 — Ordena el ciclo
Una empresa ya tiene un modelo en producción. Ordena estos pasos de un ciclo de mejora:

- a) Comparar el modelo nuevo con el actual en los mismos datos recientes
- b) Una alerta avisa que la precisión bajó
- c) Reentrenar con datos recientes
- d) Desplegar el nuevo modelo primero a un 5 % del tráfico
- e) Registrar el modelo nuevo con su versión, datos y métricas

✍️ *Mi orden:*

<details><summary><b>👀 Ver respuesta</b></summary>

**b → c → a → e → d.** La alerta (monitoreo) dispara el reentrenamiento; el modelo nuevo ("retador") se compara con el actual ("campeón") **antes** de cambiar nada; si gana, se registra con toda su información y se despliega **poco a poco**, para poder volver atrás si algo falla. Verás cada paso en las Lecciones 15.2 y 15.3.
</details>

---
## 3. 📖 Cómo funciona: empaquetar, guardar, servir

Vamos a llevar a producción un modelo que predice si un cliente de telefonía **cancelará** su contrato. Primero, los datos y el entrenamiento, como en cualquier notebook:

In [ ]:
def generar_clientes(n, semilla):
    rng = np.random.default_rng(semilla)
    df = pd.DataFrame({
        "meses_cliente":    rng.integers(1, 72, n),
        "llamadas_soporte": rng.poisson(1.5, n),
        "cargo_mensual":    rng.integers(200, 1_500, n),        # en pesos
        "contrato_anual":   rng.integers(0, 2, n),
        "plan":             rng.choice(["básico", "plus", "premium"], n, p=[.5, .3, .2]),
    })
    riesgo = (-1.5 + 0.9 * df.llamadas_soporte - 0.05 * df.meses_cliente
              + 0.002 * df.cargo_mensual - 1.8 * df.contrato_anual + 0.4 * (df.plan == "básico"))
    df["cancelo"] = (rng.random(n) < 1 / (1 + np.exp(-riesgo))).astype(int)
    return df

clientes = generar_clientes(5_000, semilla=0)
X, y = clientes.drop(columns="cancelo"), clientes["cancelo"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)

NUMERICAS = ["meses_cliente", "llamadas_soporte", "cargo_mensual", "contrato_anual"]
CATEGORICAS = ["plan"]
modelo = make_pipeline(
    ColumnTransformer([("num", StandardScaler(), NUMERICAS),
                       ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS)]),
    LogisticRegression(max_iter=1000))
modelo.fit(X_tr, y_tr)
print(f"AUC en prueba: {roc_auc_score(y_te, modelo.predict_proba(X_te)[:, 1]):.3f}")

### 📖 Regla de oro n.º 1: guarda el **pipeline completo**, no solo el modelo
El escalado y la codificación son **parte del modelo** (Lección 7.5). Si guardas solo la regresión logística, en producción alguien tendrá que **re-escribir** el preprocesamiento… y es casi seguro que lo hará un poco distinto.

▶️ Guardamos el pipeline en un archivo con `joblib` y lo cargamos como lo haría el servidor de producción:

In [ ]:
ruta = CARPETA / "modelo_cancelacion_v1.joblib"
joblib.dump(modelo, ruta)
print(f"Guardado: {ruta.name} ({ruta.stat().st_size / 1024:.1f} KB)")

modelo_en_produccion = joblib.load(ruta)        # ← esto ocurriría en otro computador
iguales = np.allclose(modelo.predict_proba(X_te), modelo_en_produccion.predict_proba(X_te))
print("¿Predice exactamente igual que en el notebook?", "✅ sí" if iguales else "❌ no")

> ⚠️ Un archivo `.joblib` (o `.pkl`) puede ejecutar código al cargarse: **nunca cargues modelos de fuentes que no sean de confianza**. Además, debe cargarse con las **mismas versiones** de las librerías con que se guardó (por eso se anotan, como verás en la Sección 5).

### 🔮 Ejercicio 3 — El desajuste silencioso
El sistema de facturación de la empresa guarda `cargo_mensual` en **miles de pesos** (1.2 en lugar de 1,200). El equipo de integración no lo nota y manda los datos así al modelo.

¿Qué pasará? ¿Dará error? ¿Cambiará mucho el % de clientes que el modelo marca como "en riesgo" (probabilidad > 0.5)?

✍️ *Mi predicción:*

In [ ]:
lote_correcto = X_te.copy()
lote_desajustado = X_te.copy()
lote_desajustado["cargo_mensual"] = lote_desajustado["cargo_mensual"] / 1000     # ¡miles de pesos!

for nombre, lote in [("Datos correctos   ", lote_correcto), ("Cargo en miles    ", lote_desajustado)]:
    p = modelo_en_produccion.predict_proba(lote)[:, 1]
    print(f"{nombre} → % marcados en riesgo: {(p > 0.5).mean():.1%} · AUC: {roc_auc_score(y_te, p):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

**No da ningún error**: para el modelo, 1.2 es un número tan válido como 1,200. Pero ahora todos los clientes parecen pagar casi nada, el modelo los ve menos propensos a irse y el porcentaje marcado en riesgo cae de forma drástica (de ≈ 31 % a ≈ 7 %); el AUC también empeora un poco (0.841 → 0.820).

Ojo con este último detalle: el AUC solo mide el **orden** de los clientes, y el orden casi se mantiene. Si el equipo de retención llama a "los que tienen probabilidad > 0.5", ahora llamará a cuatro veces menos clientes de los que debería. Las **probabilidades** importaban, no solo el orden.

Esto se llama **desajuste entre entrenamiento y producción** (*training-serving skew*): el modelo recibe en producción datos preparados de forma distinta a como los vio al entrenar. Es de los errores más comunes y más peligrosos porque es **silencioso**: el sistema sigue "funcionando".
</details>

### 📖 Regla de oro n.º 2: valida las entradas antes de predecir
La puerta de entrada del modelo debe revisar que los datos tengan sentido: columnas, tipos, rangos y categorías conocidas. Los rangos se pueden obtener de los propios datos de entrenamiento.

In [ ]:
ESQUEMA = {col: (X_tr[col].min().item(), X_tr[col].max().item()) for col in NUMERICAS}
CATEGORIAS = {"plan": set(X_tr["plan"])}

def validar(lote):
    problemas = []
    faltan = set(NUMERICAS + CATEGORICAS) - set(lote.columns)
    if faltan:
        return [f"faltan columnas: {sorted(faltan)}"]
    for col, (minimo, maximo) in ESQUEMA.items():
        fuera = ((lote[col] < minimo) | (lote[col] > maximo) | lote[col].isna()).mean()
        if fuera > 0.01:                                  # tolera hasta 1 % de casos raros
            problemas.append(f"{col}: {fuera:.0%} de valores vacíos o fuera de [{minimo}, {maximo}]")
    for col, conocidas in CATEGORIAS.items():
        nuevas = set(lote[col].dropna()) - conocidas
        if nuevas:
            problemas.append(f"{col}: categorías nunca vistas {sorted(nuevas)}")
    return problemas

def predecir(lote):
    problemas = validar(lote)
    if problemas:
        raise ValueError("Lote rechazado: " + " | ".join(problemas))
    return modelo_en_produccion.predict_proba(lote)[:, 1]

print("Rango aprendido:", ESQUEMA)

### ✋ Ejercicio 4 — Pon a prueba la puerta
Ejecuta la celda. Prueba los tres lotes: el correcto, el de "cargo en miles" y uno con un plan nuevo ("ilimitado") que marketing lanzó ayer. ¿Cuáles pasan?

✍️ *Mi predicción:*

In [ ]:
lote_plan_nuevo = X_te.head(200).copy()
lote_plan_nuevo.loc[lote_plan_nuevo.index[:50], "plan"] = "ilimitado"

for nombre, lote in [("correcto", lote_correcto), ("cargo en miles", lote_desajustado), ("plan nuevo", lote_plan_nuevo)]:
    try:
        p = predecir(lote)
        print(f"✅ {nombre}: {len(p)} predicciones")
    except ValueError as error:
        print(f"⛔ {nombre}: {error}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **Correcto:** pasa.
- **Cargo en miles:** ⛔ rechazado: el 100 % de los valores de `cargo_mensual` está fuera del rango visto (≈ 200–1,499).
- **Plan nuevo:** ⛔ rechazado por categoría nunca vista. Ojo: `OneHotEncoder(handle_unknown="ignore")` **no** habría fallado: lo habría tratado como "ningún plan" y predicho algo, en silencio. La validación convierte un error silencioso en uno **ruidoso**, que alguien puede atender.

¿Rechazar todo el lote es lo correcto? Depende del negocio: a veces se rechaza, a veces se predice con un valor por defecto y se registra una **alerta**. Lo inaceptable es no enterarse.
</details>

---
## 4. 📖 ¿Por lotes o en línea?

| | Por lotes (*batch*) | En línea (*online*, tiempo real) |
|---|---|---|
| Cuándo predice | Cada cierto tiempo (cada noche) para muchos casos | Cuando llega cada caso, en el momento |
| Ejemplo | "Lista de clientes en riesgo para que el equipo llame mañana" | "¿Apruebo esta compra con tarjeta?" |
| Velocidad que importa | El total (¿termina antes de la mañana?) | Cada respuesta (milisegundos) |
| Infraestructura | Un proceso programado | Un servicio web (API) siempre encendido |

### 🔮 Ejercicio 5 — ¿Cuánto más rápido es en lote?
Vamos a predecir 1,000 clientes **de uno en uno** (como un servicio en línea) y 1,000 clientes **en una sola llamada**. ¿Cuántas veces más rápido crees que será por cliente el lote: 2×, 10× o más de 100×?

✍️ *Mi apuesta:*

In [ ]:
muestra = X_te.head(1000)

inicio = time.perf_counter()
for i in range(len(muestra)):
    modelo_en_produccion.predict_proba(muestra.iloc[[i]])
uno_a_uno = (time.perf_counter() - inicio) / len(muestra)

inicio = time.perf_counter()
modelo_en_produccion.predict_proba(muestra)
en_lote = (time.perf_counter() - inicio) / len(muestra)

print(f"Uno a uno: {uno_a_uno * 1000:.2f} ms por cliente")
print(f"En lote  : {en_lote * 1000:.4f} ms por cliente  →  ≈ {uno_a_uno / en_lote:.0f} veces más rápido")

<details><summary><b>👀 Ver respuesta</b></summary>

En lote suele ser **cientos de veces** más rápido por cliente (el número exacto depende de tu computador). Cada llamada tiene un costo fijo (preparar el pipeline, convertir datos, etc.) que en lote se paga **una sola vez**.

Aun así, uno a uno tarda solo unos milisegundos: suficiente para un servicio en línea. La elección depende de **cuándo necesita el negocio la respuesta**, no solo de la velocidad.
</details>

### 📖 Para producción real: un servicio web en 15 líneas
En producción, un modelo en línea suele vivir detrás de una **API**. Este es un ejemplo con FastAPI (no corre en el notebook; se ejecuta como un programa aparte):

```python
# archivo: servicio.py   ·   ejecutar con:  uvicorn servicio:app
import joblib, pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI()
modelo = joblib.load("modelo_cancelacion_v1.joblib")

class Cliente(BaseModel):                     # ← validación de entradas automática
    meses_cliente: int = Field(ge=1, le=120)
    llamadas_soporte: int = Field(ge=0, le=50)
    cargo_mensual: float = Field(ge=100, le=5000)
    contrato_anual: int = Field(ge=0, le=1)
    plan: str

@app.post("/predecir")
def predecir(c: Cliente):
    p = modelo.predict_proba(pd.DataFrame([c.model_dump()]))[0, 1]
    return {"probabilidad_cancelacion": round(float(p), 3), "version_modelo": "v1"}
```

---
## 5. 📖 Las "perillas" del despliegue

En producción no ajustas hiperparámetros del modelo, sino **decisiones de operación**:

| Decisión | Opciones | Si eliges lo más "agresivo" | Si eliges lo más "conservador" | Punto de partida |
|---|---|---|---|---|
| Modo de predicción | Lotes ↔ en línea | En línea: respuesta inmediata, infraestructura más cara | Lotes: simple y barato, pero con retraso | Lotes, salvo que el negocio necesite respuesta inmediata |
| Despliegue de una versión nueva | Todo de golpe ↔ gradual | Todo de golpe: si falla, falla para todos | **Sombra** (predice sin decidir) → **canario** (5 % del tráfico) → 100 % | Sombra o canario |
| Frecuencia de reentrenamiento | Nunca ↔ continua | Se adapta rápido; riesgo de aprender ruido o datos malos | Estable, pero envejece (Lección 15.3) | Periódica + disparada por alertas |
| Validación de entradas | Permisiva ↔ estricta | Estricta: rechaza mucho, molesta | Permisiva: errores silenciosos | Estricta en lo crítico, alertas en lo demás |

### 📖 Regla de oro n.º 3: cada modelo con su ficha
Junto al archivo del modelo se guarda **todo lo necesario para entenderlo y reproducirlo**: versión, fecha, con qué datos (una huella o *hash*), con qué librerías, qué métricas obtuvo y para qué está pensado. A esto se le llama **registro de modelos** (*model registry*); herramientas como MLflow lo hacen a gran escala.

In [ ]:
import sklearn
huella_datos = hashlib.sha256(pd.util.hash_pandas_object(X_tr, index=True).values.tobytes()).hexdigest()[:12]

ficha = {
    "nombre": "modelo_cancelacion", "version": "v1", "fecha_entrenamiento": "2026-11-01",
    "archivo": ruta.name, "huella_datos_entrenamiento": huella_datos, "filas_entrenamiento": len(X_tr),
    "librerias": {"scikit-learn": sklearn.__version__, "pandas": pd.__version__},
    "metricas": {"auc_prueba": round(roc_auc_score(y_te, modelo.predict_proba(X_te)[:, 1]), 3)},
    "uso_previsto": "Priorizar llamadas de retención. NO usar para cambiar precios.",
    "esquema_entradas": {k: [float(v[0]), float(v[1])] for k, v in ESQUEMA.items()},
}
(CARPETA / "modelo_cancelacion_v1.json").write_text(json.dumps(ficha, indent=2, ensure_ascii=False))
print(json.dumps(ficha, indent=2, ensure_ascii=False))

### ✋ Ejercicio 6 — ¿Para qué sirve cada campo?
Seis meses después, el modelo empieza a fallar. ¿Qué campo de la ficha te ayuda a responder cada pregunta?
1. "¿Con qué datos se entrenó exactamente? ¿Son los mismos que tenemos guardados?"
2. "Lo cargué en el servidor nuevo y da error. ¿Por qué?"
3. "Marketing quiere usarlo para decidir descuentos. ¿Deberíamos?"

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. `huella_datos_entrenamiento` (y `filas_entrenamiento`): si recalculas la huella de tus datos guardados y no coincide, **no son los mismos**. Lo verás a fondo en la Lección 15.3.
2. `librerias`: probablemente el servidor nuevo tiene otra versión de scikit-learn.
3. `uso_previsto`: el modelo se pensó para priorizar llamadas, no para precios. Usarlo para otra cosa requiere evaluarlo de nuevo (y pensar en su impacto: Lección 15.4).
</details>

---
## 6. 📖 Errores comunes y pruebas de comportamiento

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| El modelo en producción rinde mucho peor que en el notebook, desde el día 1 | *Training-serving skew*: preprocesamiento o unidades distintas | Guardar el pipeline completo; validar entradas; comparar distribuciones (15.2) |
| Error al cargar el modelo en el servidor | Versiones de librerías distintas | Fijar versiones (`requirements.txt`), contenedores (Docker) |
| Nadie sabe qué modelo está funcionando | Sin registro de modelos | Ficha + versión en cada predicción |
| El modelo rinde bien en promedio pero hace cosas absurdas en casos concretos | Faltan pruebas de comportamiento | Pruebas direccionales y de invariancia (abajo) |
| Una versión nueva rompe todo | Despliegue de golpe | Sombra, canario, plan de vuelta atrás (*rollback*) |

### 📖 Pruebas de comportamiento
Igual que el software tiene pruebas, un modelo debería tenerlas. Dos tipos muy útiles:
- **Direccionales:** "si un cliente llama **más** a soporte, su riesgo **no** debería bajar".
- **De invariancia:** "cambiar algo irrelevante no debería cambiar la predicción".

### 🔮 Ejercicio 7 — ¿Pasará las pruebas?
¿Crees que nuestro modelo pasará las dos pruebas de abajo? ¿Por qué?

✍️ *Mi predicción:*

In [ ]:
def prueba_direccional(modelo, base, columna, delta, debe):
    alterado = base.copy(); alterado[columna] += delta
    cambio = modelo.predict_proba(alterado)[:, 1] - modelo.predict_proba(base)[:, 1]
    ok = (cambio >= 0).all() if debe == "subir" else (cambio <= 0).all()
    print(f"{'✅' if ok else '❌'} +{delta} en {columna}: el riesgo debe {debe} → cambio medio {cambio.mean():+.3f}")

def prueba_invariancia(modelo, base, columna, valor_nuevo):
    alterado = base.copy(); alterado[columna] = valor_nuevo
    dif = np.abs(modelo.predict_proba(alterado)[:, 1] - modelo.predict_proba(base)[:, 1]).max()
    print(f"{'✅' if dif < 1e-9 else '❌'} cambiar {columna}: diferencia máxima {dif:.4f}")

base = X_te.head(300).copy()
prueba_direccional(modelo_en_produccion, base, "llamadas_soporte", 2, "subir")
prueba_direccional(modelo_en_produccion, base, "meses_cliente", 12, "bajar")
base["id_cliente"] = range(len(base))                  # una columna que el modelo no usa
prueba_invariancia(modelo_en_produccion, base, "id_cliente", 999)

<details><summary><b>👀 Ver respuesta</b></summary>

Pasa las tres:
- Más llamadas a soporte → el riesgo **sube** en todos los clientes.
- Más antigüedad → el riesgo **baja**.
- Cambiar `id_cliente` no cambia nada: el `ColumnTransformer` solo usa las columnas que declaramos (cualquier columna extra se descarta).

En un modelo lineal estas pruebas casi siempre pasan; en modelos más flexibles (árboles, redes) **pueden fallar** en zonas con pocos datos, y entonces descubres comportamientos absurdos antes que tus clientes.
</details>

---
## 7. 📋 Ficha: lista de verificación antes de desplegar

| ✔️ | Pregunta |
|---|---|
| ☐ | ¿Guardé el **pipeline completo** (preprocesamiento + modelo)? |
| ☐ | ¿Hay **validación de entradas** (columnas, tipos, rangos, categorías)? |
| ☐ | ¿El modelo tiene **ficha**: versión, datos, librerías, métricas, uso previsto? |
| ☐ | ¿Pasa **pruebas de comportamiento** (direccionales, invariancia)? |
| ☐ | ¿Elegí **lotes o en línea** según cuándo necesita el negocio la respuesta? |
| ☐ | ¿Hay un plan de **despliegue gradual** y de **vuelta atrás**? |
| ☐ | ¿Qué se va a **monitorear** y quién recibe las alertas? (Lección 15.2) |

## 8. 🧠 Resumen en 5 puntos
1. **MLOps** = prácticas para que el ciclo datos → modelo → despliegue → monitoreo → reentrenamiento sea confiable y repetible.
2. El modelo es **la parte pequeña**: datos, validación, versiones, monitoreo y despliegue son la mayor parte del trabajo.
3. Guarda el **pipeline completo** y **valida las entradas**: el *training-serving skew* es silencioso y muy común.
4. Cada modelo necesita una **ficha** (versión, datos, librerías, métricas, uso previsto) y **pruebas de comportamiento**.
5. Elige **lotes** o **en línea** según cuándo se necesita la respuesta, y despliega **gradualmente** con plan de vuelta atrás.

---
## 9. 🏋️ Ejercicios finales

### 9.1 Repaso rápido (opción múltiple)

**1.** ¿Qué es el *training-serving skew*?
- a) Que el modelo tarda en entrenar · b) Que los datos en producción se preparan distinto que en entrenamiento · c) Que el modelo sobreajusta · d) Que el servidor es lento

**2.** ¿Por qué conviene guardar el pipeline completo y no solo el modelo?
- a) Ocupa menos espacio · b) Para que el preprocesamiento en producción sea idéntico al del entrenamiento · c) Es obligatorio en scikit-learn · d) Para entrenar más rápido

**3.** Un banco necesita decidir si aprueba una compra con tarjeta mientras el cliente está en la caja. ¿Qué modo de predicción?
- a) Por lotes cada noche · b) En línea · c) Da igual · d) Ninguno: no se puede usar ML

**4.** Un despliegue "canario" consiste en…
- a) Probar el modelo con datos de pájaros · b) Enviar primero una pequeña parte del tráfico al modelo nuevo · c) Desplegar todo de golpe · d) Borrar el modelo viejo

**5.** "Si aumento el ingreso de un solicitante, su probabilidad de aprobación no debería bajar." Esto es una prueba…
- a) De invariancia · b) Direccional · c) De carga · d) De fuga de datos

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** 2. **b)** 3. **b)** 4. **b)** (como los canarios que los mineros llevaban para detectar gases antes que las personas) 5. **b)**
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** El equipo de datos entrena el modelo en Python. El equipo de sistemas propone **reescribir** el preprocesamiento en Java "porque es más rápido". ¿Qué riesgo ves? ¿Cómo lo mitigarías si no hay más remedio?

**B. Caso de análisis 💼** Un hospital quiere desplegar un modelo que ordena qué pacientes de urgencias atender primero. Diseña el despliegue en 3 etapas, de menor a mayor riesgo, usando el vocabulario de la lección.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Riesgo de *training-serving skew*: cualquier diferencia mínima (orden de las categorías, redondeos, cómo se tratan los vacíos) cambia las predicciones en silencio. Mitigación: un conjunto de **casos de prueba** (entradas y probabilidades esperadas generadas con el pipeline de Python) que la versión Java debe reproducir exactamente antes de desplegarse, y monitorear que las distribuciones de predicciones coincidan. Alternativas: exportar el pipeline a un formato estándar (por ejemplo, ONNX) o servirlo desde Python.

**B.** Un diseño posible:
1. **Modo sombra** durante semanas: el modelo predice para cada paciente, pero **nadie** ve sus resultados; se comparan después con las decisiones del personal y con lo que pasó.
2. **Apoyo a la decisión** en un área piloto: el personal ve la sugerencia, pero decide; se registran los desacuerdos y se revisan.
3. **Ampliación gradual** a más áreas, con monitoreo continuo, una ficha clara del uso previsto, revisión de errores por grupos de pacientes (Lección 15.4) y la posibilidad de apagarlo de inmediato. El modelo nunca decide solo en un caso de este riesgo (Lección 15.6).
</details>

### 9.3 🧩 Práctica integradora: "Versión 2 a producción"

El equipo entrenó una **versión 2** del modelo que, además, usa un campo nuevo: `uso_datos_gb` (gigas usados al mes). Tu trabajo es decidir si está lista para producción, en 5 pasos cortos. **No avances sin revisar cada respuesta.**

**Paso 1 ▶️ — Entrena la versión 2**

In [ ]:
rng = np.random.default_rng(1)
X_tr2, X_te2 = X_tr.copy(), X_te.copy()
X_tr2["uso_datos_gb"] = rng.gamma(2, 5, len(X_tr2)).round(1)
X_te2["uso_datos_gb"] = rng.gamma(2, 5, len(X_te2)).round(1)

NUMERICAS_V2 = NUMERICAS + ["uso_datos_gb"]
modelo_v2 = make_pipeline(
    ColumnTransformer([("num", StandardScaler(), NUMERICAS_V2),
                       ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS)]),
    LogisticRegression(max_iter=1000)).fit(X_tr2, y_tr)

for nombre, m, Xp in [("v1", modelo_en_produccion, X_te), ("v2", modelo_v2, X_te2)]:
    print(f"{nombre}: AUC en prueba = {roc_auc_score(y_te, m.predict_proba(Xp)[:, 1]):.3f}")

**Paso 2 🔮 — ¿Vale la pena?** Mira los AUC. ¿La versión 2 es mejor? (Pista: mira cómo se generó `uso_datos_gb`.)

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Los AUC son **iguales** (0.841 en ambos casos, al menos hasta el tercer decimal). No es raro: `uso_datos_gb` se generó **al azar**, sin relación con la cancelación. La versión 2 añade un campo que hay que recolectar, validar y monitorear… **sin aportar nada**. En producción, cada campo extra es un costo y un punto de fallo más. Decisión razonable: **no desplegarla**.
</details>

**Paso 3 ✏️ — Supón que sí aportara.** Antes de desplegarla habría que actualizar la puerta de entrada. Completa `ESQUEMA_V2` con el rango de `uso_datos_gb` aprendido de `X_tr2` y comprueba que un lote **sin** esa columna es rechazado.

In [ ]:
ESQUEMA_V2 = {col: (X_tr2[col].min().item(), X_tr2[col].max().item()) for col in NUMERICAS_V2}   # ✏️ (ya está; revísalo)

def validar_v2(lote):
    faltan = set(NUMERICAS_V2 + CATEGORICAS) - set(lote.columns)
    return [f"faltan columnas: {sorted(faltan)}"] if faltan else []

print("Lote viejo (sin uso_datos_gb):", validar_v2(X_te) or "✅ válido")
print("Lote nuevo                    :", validar_v2(X_te2) or "✅ válido")
print("Rango de uso_datos_gb:", ESQUEMA_V2["uso_datos_gb"])

<details><summary><b>👀 Ver respuesta</b></summary>

El lote viejo es rechazado por la columna faltante; el nuevo es válido. Esto importa en la transición: mientras **los sistemas que envían datos** no se actualicen, la versión 2 no puede recibir peticiones. Por eso un cambio de versión con campos nuevos se coordina con los equipos que producen esos datos (y se despliega primero en **sombra**).
</details>

**Paso 4 ▶️ — Pruebas de comportamiento de la v2.** ¿Siguen pasando las pruebas direccionales?

In [ ]:
base2 = X_te2.head(300).copy()
prueba_direccional(modelo_v2, base2, "llamadas_soporte", 2, "subir")
prueba_direccional(modelo_v2, base2, "meses_cliente", 12, "bajar")
prueba_direccional(modelo_v2, base2, "uso_datos_gb", 10, "bajar")    # hipótesis de negocio: más uso = más enganchado

<details><summary><b>👀 Ver respuesta</b></summary>

Las tres pasan, pero mira la tercera: el cambio medio es **−0.003**, prácticamente cero. El campo no tiene relación real con la cancelación, así que el coeficiente aprendido es casi 0 y su **signo depende del ruido de la muestra**: con otros datos de entrenamiento la prueba podría fallar. Una prueba que pasa "de milagro" es otra señal de que **ese campo no aporta**.
</details>

**Paso 5 ✋ — Tu decisión.** Escribe 3 frases para el gerente: ¿se despliega la v2? ¿Por qué? ¿Qué harías en su lugar?

✍️ *Mi decisión:*

<details><summary><b>👀 Ver respuesta</b></summary>

Un ejemplo:

> «Recomiendo **no desplegar** la versión 2: predice igual que la actual (AUC ≈ 0.84 en ambas) y exige recolectar un dato nuevo que no aporta información. Mantener la versión 1 es más simple y barato, y cada campo extra es un punto más donde algo puede fallar. En su lugar, propongo invertir en monitorear la versión 1 (Lección 15.2) y explorar variables con relación real con la cancelación, como quejas recientes o cambios de plan.»
</details>

---
## 📚 Para profundizar
- 📰 **Artículo:** Sculley et al. (2015), *Hidden Technical Debt in Machine Learning Systems* (NeurIPS).
- 📘 **Libro:** Chip Huyen, *Designing Machine Learning Systems* (O'Reilly, 2022): el mejor panorama práctico de ML en producción.
- 📰 **Lectura:** Google, *Rules of Machine Learning* (Martin Zinkevich): 43 reglas prácticas, gratis en línea.
- 📰 **Pruebas:** Ribeiro et al. (2020), *Beyond Accuracy: Behavioral Testing of NLP Models with CheckList*.
- 🛠️ **Herramientas:** MLflow (registro de modelos), FastAPI (servicios), Docker (entornos reproducibles), Evidently (monitoreo).

---
⬅️ **Anterior:** [14.6 Proyecto del módulo: El dron repartidor 🚁📦](../m14-refuerzo/14.6-proyecto-del-modulo.md) · ➡️ **Siguiente:** [15.2 Monitoreo: deriva de datos y de concepto](15.2-monitoreo-deriva-de-datos-y-concepto.ipynb)